# Objectives

- Convert the mrc file containing 4DSTEM data into a h5 file. Note that the data is originally stored as 3D; we need information about the real-space (x and y) dimensions to restore the 4D structure of the data. (Idk why the microscope makes it 3D.)
- Shift up the values of intensities such that there are no negative or zero intensities. These are both unphysical and cause errors for log plots, so we want to get rid of them.
- Save the 4D intensity data as a new file. Plot some sample diffraction patterns for sanity check. (A later script will calculate a brightfield image, which we will compare to the HAADF image of the wire to check consistency. If we chose incorrect reshaping parameters, the brightfield would be obviously incorrect.)

In [ ]:
import mrcfile
import numpy as np
import py4DSTEM
print(py4DSTEM.__version__)


In [ ]:
input_file_path = 'folder/file.mrc'
output_file_path = 'file.h5'
output_file_path

# Reshape data into 4D array

In [ ]:
Scan_X = 159  # Replace with your desired value of 'Scan_X' 
Scan_Y = 122  # Replace with your desired value of 'Scan_Y' (ensure Scan_X * Scan_Y = n)

with mrcfile.open(input_file_path, permissive=True) as mrc:
        original_data = mrc.data.copy()
        print("Original data shape:", original_data.shape)  # (n, height, width)
        
        n, height, width = original_data.shape

        # Ensure the reshaping is valid
        if n != Scan_Y * Scan_X:
            raise ValueError(f"Invalid dimensions: {Scan_Y} * {Scan_X} ≠ {n}")

        # Reshape to (Scan_Y, Scan_X, height, width)
        data_reshaped = original_data.reshape((Scan_Y, Scan_X, height, width))
        print("Reshaped data shape:", data_reshaped.shape)


# Shift minimum above zero

Instead of adding the minimum intensity to all elements at once, I will iterate over the first axis and update the array in-place. This avoids the error `MemoryError: Unable to allocate 37.9 GiB for an array with shape (122, 159, 512, 512) and data type float64`

In [ ]:
min_I = np.min(data_reshaped)
print(min_I)
if min_I <= 0:
    for i, row in enumerate(data_reshaped):
        data_reshaped[i] = row - min_I + 1

    print(np.min(data_reshaped))

# Saving as dataset/Datacube readable with py4DSTEM

In [ ]:
dataset = py4DSTEM.DataCube(data_reshaped)
py4DSTEM.save(output_file_path, dataset, mode='overwrite')
print("h5 file successfully created as", output_file_path)


# Look at an example diffraction pattern

In [ ]:
from py4DSTEM import show

sample_diffraction = dataset[39, 46]

show(sample_diffraction)